# Anarkali Web — train the web expert on real web pages

One notebook, start to finish, on Colab or Kaggle (GPU runtime: T4 is enough).

1. Real pages come from **Common Crawl** (built by `Anarkali_Data_Build.ipynb`, or here if `DATA_FROM = ""`).
2. The model reads each page's **structure** (menus, main content, forms, headings) and answers questions any
   site has: `page_type` (22 kinds), `page_status`, `language` and `published_date`.
3. Labels come from free real sources (schema.org checked against og:type and the URL, the HTTP status,
   `<html lang>` agreeing with Common Crawl's detector). The model never sees them.
4. It is scored on the test split (sites it never saw) against simple baselines.
5. The model is exported to ONNX, and `best.pt` is saved too, so it is never lost.

Every cell's output is also written to `anarkali-web-log.txt` and uploaded to HF after every cell.
Checkpoints and data go to a **private** Hugging Face repo as soon as they exist. Add an `HF_TOKEN` secret
(write access) or paste the token when asked; it is never saved in the notebook.

Kaggle: Settings → Internet **On**, Accelerator **GPU**, then Save Version → Save & Run All.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/ToufiqQureshi/anarkali.git"
BRANCH = "main"

RUN_NAME = "run3"               # results go to HF_REPO/<RUN_NAME>/ and the run log in RUNS.md
HF_REPO = "toufiqqureshi651/anarkali-web"   # private repo for checkpoints and data; needs an HF token
INIT_FROM = ""                  # a checkpoint in HF_REPO to continue from; "" = start from BASE_MODEL
                                # (run3 changed page_state, so earlier checkpoints do not carry over)

BASE_MODEL = "jhu-clsp/ettin-encoder-150m"   # must match the INIT_FROM checkpoint
DATA_FROM = "run3/web-expert-v1-checked"   # labels re-checked by Qwen3 (Anarkali_Label_Check.ipynb); "" = build here
CRAWL = "CC-MAIN-2026-39"       # fixed crawl so runs are comparable
WARCS = 12                      # used only when DATA_FROM is "": each WARC holds ~30-40k HTML pages
ERROR_WARCS = 3

EPOCHS = 2                      # run2: the best epoch came early; 2 is enough
BATCH_SIZE = 16
MAX_TOKENS = 512
ENCODER_LR = 3e-5

JULIA_BASELINE = False          # also score SupersonicLabs/Julia-1 on the same test split

## 2. Setup

In [ ]:
import os, subprocess, sys, json, shutil, datetime, traceback
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("/content")
REPO = WORK / "anarkali"
LOG = WORK / "anarkali-web-log.txt"   # every cell's output, cell by cell

def _log(text):
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(text)

class _Tee:
    """Send what a cell prints to the notebook and to LOG."""
    def __init__(self, stream):
        self.stream = stream
    def write(self, text):
        self.stream.write(text)
        _log(text)
        return len(text)
    def flush(self):
        self.stream.flush()
    def __getattr__(self, name):
        return getattr(self.stream, name)

def _cell_start(info=None, title=None):
    for name in ("stdout", "stderr"):
        if not isinstance(getattr(sys, name), _Tee):
            setattr(sys, name, _Tee(getattr(sys, name)))
    if title is None:
        title = next((l for l in info.raw_cell.strip().splitlines() if l.strip()), "")[:90]
    _log(f"\n===== {datetime.datetime.now():%Y-%m-%d %H:%M:%S} | {title} =====\n")

def _cell_end(result):
    if result.error_in_exec is not None:
        _log("ERROR:\n" + "".join(traceback.format_exception(result.error_in_exec)))
    save_log()

def save_log():
    """Copy LOG to HF_REPO/<RUN_NAME>/notebook-log.txt; never raises."""
    if "hf" not in globals():
        return
    try:
        hf.upload_file(repo_id=HF_REPO, path_or_fileobj=str(LOG), path_in_repo=f"{RUN_NAME}/notebook-log.txt")
    except Exception as exc:
        print("log upload failed:", exc, file=sys.__stdout__)

ip = get_ipython()
for event, fn in (("pre_run_cell", _cell_start), ("post_run_cell", _cell_end)):
    for old in [f for f in ip.events.callbacks[event] if getattr(f, "__name__", "") == fn.__name__]:
        ip.events.unregister(event, old)
    ip.events.register(event, fn)
_cell_start(title="2. Setup")

def sh(cmd):
    """Run a shell command, streaming its output to the notebook and LOG; raise if it fails."""
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=str(REPO) if REPO.exists() else None,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors="replace",
                            bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait():
        raise subprocess.CalledProcessError(proc.returncode, cmd)

if not REPO.exists():
    sh(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO}")
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))   # the kernel started before pip install -e, so it cannot see the package otherwise
sh('pip install -q -e ".[train,onnx]" warcio requests datasets onnx')

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to GPU")
print("log file:", LOG)

def _hf_token():
    token = os.environ.get("HF_TOKEN")
    for getter in ("google.colab.userdata", "kaggle_secrets"):
        if token:
            break
        try:
            if getter == "google.colab.userdata":
                from google.colab import userdata
                token = userdata.get("HF_TOKEN")
            else:
                from kaggle_secrets import UserSecretsClient
                token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            pass
    if not token:
        from getpass import getpass
        token = getpass("Hugging Face write token (not saved): ").strip()
    return token

from huggingface_hub import HfApi, hf_hub_download
HF_TOKEN = _hf_token()
hf = HfApi(token=HF_TOKEN)
print("Hugging Face user:", hf.whoami()["name"])
hf.create_repo(HF_REPO, private=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA = REPO / "artifacts" / "web-expert-v1"
RUN = REPO / "artifacts" / "web-run"
RELEASE = REPO / "artifacts" / "release-web"

## 3. Build the dataset from Common Crawl

It takes a few minutes per WARC. Progress prints every 2,000 pages.

In [ ]:
if DATA_FROM:
    from huggingface_hub import snapshot_download
    got = snapshot_download(HF_REPO, repo_type="model", allow_patterns=[f"{DATA_FROM}/*"], token=HF_TOKEN)
    if DATA.exists():
        shutil.rmtree(DATA)
    shutil.copytree(Path(got) / DATA_FROM, DATA)
    print("dataset downloaded from", f"{HF_REPO}/{DATA_FROM}")
else:
    sh(f"python scripts/build_web_expert.py --crawl {CRAWL} --warcs {WARCS} --error-warcs {ERROR_WARCS} "
       f"--workers {max(2, os.cpu_count())} --output {DATA}")
    # Save the dataset now, so a disconnect during training does not lose it.
    hf.upload_folder(repo_id=HF_REPO, folder_path=str(DATA), path_in_repo=f"{RUN_NAME}/web-expert-v1")
    print("dataset uploaded to", f"{HF_REPO}/{RUN_NAME}/web-expert-v1")

## 4. What the data looks like

In [ ]:
manifest = json.loads((DATA / "manifest.json").read_text())
print("crawl:", manifest.get("crawl"), "| pages:", manifest.get("collection_stats"))
print("dataset:", manifest.get("dataset"), "| label check:", manifest.get("label_check", {}).get("model"))
for split, info in manifest["split_counts"].items():
    print(f"\n{split}: {info['decision_cases']} decisions from {info['source_groups']} sites")
    for label, n in info["labels"].items():
        print(f"   {label:30s} {n}")

first = json.loads(next(open(DATA / "train.jsonl")))
print("\nExample state the model reads:")
print(json.dumps(first["state"], ensure_ascii=False)[:600])

## Preflight (~5 min)

Before the long training, check what could fail after it: every row of every split must fit in
`MAX_TOKENS`, and the whole chain (train, evaluate, ONNX export, `Engine`) must run on a tiny copy of
the data. If anything here fails, fix it before spending GPU hours. The tiny model is thrown away.


In [ ]:
import hashlib
from collections import Counter
from transformers import AutoTokenizer
from anarkali.packing import pack_row

# 1. Every row of every split must pack at MAX_TOKENS (this is what crashed run2's first attempt).
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
too_long = Counter()
largest = 0
manifest_all = json.loads((DATA / "manifest.json").read_text())
for split in ("train", "development", "calibration", "test"):
    keep, dropped = [], 0
    for line in open(DATA / f"{split}.jsonl", encoding="utf-8"):
        row = json.loads(line)
        try:
            ids_, _, meta = pack_row(row, tokenizer, MAX_TOKENS)
            largest = max(largest, meta["input_tokens"] - (meta["state_tokens"] - meta["state_tokens_dropped"]))
            keep.append(line)
        except ValueError as exc:
            if split == "train":   # a handful of rows with very long options: drop them from training only
                dropped += 1
                print("  dropped from train (cannot pack):", row["case_id"], str(exc))
            else:
                too_long[(split, row["case_id"].rsplit("::", 1)[-1], str(exc))] += 1
    if dropped:
        assert dropped <= 0.001 * (len(keep) + dropped), "too many train rows cannot pack"
        (DATA / f"{split}.jsonl").write_text("".join(keep), encoding="utf-8")
        info = manifest_all["split_counts"][split]
        info["decision_cases"] = len(keep)
        info["sha256"] = hashlib.sha256((DATA / f"{split}.jsonl").read_bytes()).hexdigest()
        info["dropped_cannot_pack"] = dropped
        (DATA / "manifest.json").write_text(json.dumps(manifest_all, indent=2, ensure_ascii=False))
print("largest question+options size:", largest, "tokens of", MAX_TOKENS)
if too_long:
    for k, n in too_long.most_common(10):
        print("  cannot pack:", k, n)
    raise RuntimeError("some rows do not fit MAX_TOKENS; raise MAX_TOKENS or shorten the options")

# 2. A tiny copy of the data with every decision type, and the full chain on it.
MINI = WORK / "preflight-data"
MINI_RUN = WORK / "preflight-run"
for d in (MINI, MINI_RUN):
    shutil.rmtree(d, ignore_errors=True)
MINI.mkdir()
mini_manifest = json.loads((DATA / "manifest.json").read_text())
for split in ("train", "development", "calibration", "test"):
    kept, per_key = [], Counter()
    for line in open(DATA / f"{split}.jsonl", encoding="utf-8"):
        k = json.loads(line)["case_id"].rsplit("::", 1)[-1]
        if per_key[k] < 12:
            per_key[k] += 1
            kept.append(line)
    (MINI / f"{split}.jsonl").write_text("".join(kept), encoding="utf-8")
    mini_manifest["split_counts"][split]["sha256"] = hashlib.sha256((MINI / f"{split}.jsonl").read_bytes()).hexdigest()
    print(split, dict(per_key))
(MINI / "manifest.json").write_text(json.dumps(mini_manifest))

pre_init = ""
if INIT_FROM:
    pre_init = f"--init-checkpoint {hf_hub_download(HF_REPO, INIT_FROM, token=HF_TOKEN)}"
sh(f"python scripts/train_anarkali.py --data {MINI} --output {MINI_RUN} --architecture packed "
   f"--model {BASE_MODEL} --packed-max-tokens {MAX_TOKENS} --epochs 1 --batch-size {BATCH_SIZE} "
   f"--encoder-lr {ENCODER_LR} --selection-metric accuracy_then_ce --schedule linear --warmup-ratio 0.06 "
   f"--device {DEVICE} {pre_init}")
sh(f"python scripts/evaluate_checkpoint.py --checkpoint {MINI_RUN}/best.pt --data {MINI} --device {DEVICE} "
   f"--latency-samples 5 --output {MINI_RUN}")
pre_report = json.loads((MINI_RUN / "test-report.json").read_text())
assert (MINI_RUN / "anarkali-test.jsonl").exists()
sh(f"python scripts/export_onnx.py --checkpoint {MINI_RUN}/best.pt --output {WORK / 'preflight-release'} "
   f"--data {MINI} --name preflight --temperature-by-type '{json.dumps(pre_report['temperature_by_type'])}' --no-int8")
from anarkali.engine import Engine
from anarkali.webpage import page_state, PAGE_TYPES, PAGE_TYPE_QUESTION
print(Engine.load(str(MINI_RUN / "best.pt")).predict(
    page_state("<html><body><nav><a href='/'>Home</a></nav><main><h1>Opening hours</h1><p>"
               + "The library is open every day from nine to five. " * 6 + "</p></main></body></html>",
               "https://library.example/hours"),
    {"page_type": {"type": "choice", "instructions": PAGE_TYPE_QUESTION, "criteria": PAGE_TYPES}})["answers"])
for d in (MINI_RUN, WORK / "preflight-release"):
    shutil.rmtree(d, ignore_errors=True)
print("PREFLIGHT OK: the long training can start")


## 5. Train

Progress prints live: one line every 25 steps and one per epoch. The best epoch is saved as `best.pt` after each epoch.

In [ ]:
init_flag = ""
if INIT_FROM:
    init_path = hf_hub_download(HF_REPO, INIT_FROM, token=HF_TOKEN)
    init_flag = f"--init-checkpoint {init_path}"
    print("starting from", f"{HF_REPO}/{INIT_FROM}")
import threading

def _watch_checkpoint(stop, every=120):
    """Upload best.pt and training.json whenever best.pt changes, so a timeout keeps the last best epoch."""
    seen = None
    while not stop.wait(every):
        best = RUN / "best.pt"
        if not best.exists() or best.stat().st_mtime == seen:
            continue
        seen = best.stat().st_mtime
        if stop.wait(15) or best.stat().st_mtime != seen:   # still being written
            seen = None
            continue
        try:
            hf.upload_file(repo_id=HF_REPO, path_or_fileobj=str(best), path_in_repo=f"{RUN_NAME}/web-run/best.pt")
            for name in ("training.json",):
                if (RUN / name).exists():
                    hf.upload_file(repo_id=HF_REPO, path_or_fileobj=str(RUN / name),
                                   path_in_repo=f"{RUN_NAME}/web-run/{name}")
            print("checkpoint saved to HF", flush=True)
        except Exception as exc:
            print("checkpoint upload failed (will retry):", exc, flush=True)
            seen = None

_stop = threading.Event()
_watcher = threading.Thread(target=_watch_checkpoint, args=(_stop,), daemon=True)
_watcher.start()
try:
    sh(f"python scripts/train_anarkali.py --data {DATA} --output {RUN} --architecture packed "
       f"--model {BASE_MODEL} --packed-max-tokens {MAX_TOKENS} --epochs {EPOCHS} --batch-size {BATCH_SIZE} "
       f"--encoder-lr {ENCODER_LR} --selection-metric accuracy_then_ce --schedule linear --warmup-ratio 0.06 "
       f"--log-every 100 --device {DEVICE} {init_flag}")
finally:
    _stop.set()
    _watcher.join()
hf.upload_file(repo_id=HF_REPO, path_or_fileobj=str(RUN / "best.pt"), path_in_repo=f"{RUN_NAME}/web-run/best.pt")
print("best.pt uploaded to", f"{HF_REPO}/{RUN_NAME}/web-run/best.pt")

## 6. Test on unseen sites

Temperature is fitted on the calibration split, then the test split is scored once.

In [ ]:
sh(f"python scripts/evaluate_checkpoint.py --checkpoint {RUN}/best.pt --data {DATA} --device {DEVICE} --output {RUN}")
report = json.loads((RUN / "test-report.json").read_text())

In [ ]:
from collections import Counter, defaultdict

def key(case_id):
    return case_id.rsplit("::", 1)[-1]

def load(path):
    return [json.loads(line) for line in open(path) if line.strip()]

def gold(row):
    return row["candidates"][max(range(len(row["target"])), key=row["target"].__getitem__)]["id"]

train = [r for r in load(DATA / "train.jsonl") if r["workflow"] == "web"]
test_rows = {r["case_id"]: r for r in load(DATA / "test.jsonl")}
preds = load(RUN / "anarkali-test.jsonl")

# Baseline per decision: the most common train label when every row has the same options
# (page_type, page_status), else a random pick among the row's options (language, published_date).
option_sets, labels = defaultdict(set), defaultdict(Counter)
for r in train:
    option_sets[key(r["case_id"])].add(tuple(c["id"] for c in r["candidates"]))
    labels[key(r["case_id"])][gold(r)] += 1
majority = {q: labels[q].most_common(1)[0][0] for q, sets in option_sets.items() if len(sets) == 1}

stats = defaultdict(lambda: {"n": 0, "model": 0, "baseline": 0.0, "sure_n": 0, "sure_ok": 0})
for p in preds:
    q = key(p["case_id"])
    s = stats[q]
    s["n"] += 1
    ok = p["choice"] == p["expected"]
    s["model"] += ok
    if q in majority:
        s["baseline"] += p["expected"] == majority[q]
    else:
        target = test_rows[p["case_id"]]["target"]
        s["baseline"] += sum(t > 0 for t in target) / len(target)
    if max(p["probabilities"].values()) >= 0.9:
        s["sure_n"] += 1
        s["sure_ok"] += ok

print(f"{'decision':15s} {'n':>5s} {'model':>7s} {'baseline':>9s}   {'confident>=0.9: coverage / accuracy':s}")
for q, s in sorted(stats.items()):
    base = f"majority {majority[q]}" if q in majority else "random pick"
    sure = f"{s['sure_n']/s['n']:.0%} / {s['sure_ok']/s['sure_n']:.1%}" if s["sure_n"] else "-"
    print(f"{q:15s} {s['n']:5d} {s['model']/s['n']:7.1%} {s['baseline']/s['n']:8.1%}   {sure}   ({base})")

per_type = defaultdict(lambda: [0, 0])
for p in preds:
    if key(p["case_id"]) == "page_type":
        per_type[p["expected"]][0] += 1
        per_type[p["expected"]][1] += p["choice"] == p["expected"]
print("\npage_type by true label:", {k: f"{ok}/{n}" for k, (n, ok) in sorted(per_type.items())})
print("\ncalibrated by type:", json.dumps(report["test_calibrated_by_type"]["all"], indent=1))
print("order-reversal flips:", report["order_reversal_argmax_change_fraction"])
print("latency:", report["latency"])

**Read the table like this:** the model must beat the baseline clearly. The most important
column is the last one: when the model is ≥0.9 sure, that accuracy should be 95%+. Those are the
pages it can handle on its own, and everything else goes to an LLM or a person.

### Julia-1 on the same test split

A general decision model of the same size class (SupersonicLabs/Julia-1, Apache-2.0). If Anarkali does
not clearly beat it on these web decisions, the specialist is not worth shipping.

In [ ]:
from collections import Counter
try:
    julia_result = None
    if JULIA_BASELINE:
        from huggingface_hub import snapshot_download
        julia_dir = snapshot_download("SupersonicLabs/Julia-1", local_dir=str(WORK / "Julia-1"))
        sh(f"pip install -q -e {julia_dir}")
        sys.path.insert(0, julia_dir)   # the julia/ package lives at the repo root
        from julia import load_model
        julia = load_model(julia_dir, device=DEVICE, strict_encoding=True, max_length=2048, head_length=512)

        def julia_question(row):
            kind = row.get("question_type", "choice")
            ids = [c["id"] for c in row["candidates"]]
            criteria = {c["id"]: c["text"] for c in row["candidates"]}
            text = row["question"].removeprefix("True or false: ")
            return {"type": kind, "instructions": text, "criteria": criteria}, ids

        julia_stats = defaultdict(lambda: [0, 0, 0])   # n, correct, skipped
        julia_errors = Counter()
        for row in test_rows.values():
            q = key(row["case_id"])
            spec, ids = julia_question(row)
            expected = ids[max(range(len(ids)), key=row["target"].__getitem__)]
            julia_stats[q][0] += 1
            if len(ids) > 20:
                julia_stats[q][2] += 1
                continue
            try:
                answer = julia.predict(state=json.dumps(row["state"], ensure_ascii=False), questions={"q": spec})["answers"]["q"]
            except Exception as exc:
                julia_stats[q][2] += 1
                julia_errors[repr(exc)[:300]] += 1
                continue
            picked = answer.get("choice") if spec["type"] == "choice" else ("true" if answer["noul"] >= 0.5 else "false")
            julia_stats[q][1] += picked == expected
        print(f"{'decision':12s} {'Anarkali':>9s} {'Julia-1':>9s}  skipped")
        for q, (n, ok, skipped) in sorted(julia_stats.items()):
            print(f"{q:12s} {stats[q]['model']/stats[q]['n']:9.1%} {ok/n:9.1%}  {skipped}")
        for err, n in julia_errors.most_common(3):
            print(f"Julia-1 error ({n} rows): {err}")
        # a baseline that skipped most rows is not a baseline: do not save it as one
        if sum(v[2] for v in julia_stats.values()) > 0.1 * sum(v[0] for v in julia_stats.values()):
            raise RuntimeError("Julia-1 skipped over 10% of the test rows; see the errors above")
        julia_result = {q: v[1] / (v[0] - v[2]) for q, v in julia_stats.items()}
    else:
        julia_result = None
except Exception as exc:   # optional step: never block saving
    print('Julia-1 baseline skipped:', repr(exc))


## 7. Try it on any HTML

In [ ]:
try:
    from anarkali.engine import Engine
    from anarkali.webpage import page_state, PAGE_TYPES, PAGE_TYPE_QUESTION, PAGE_STATUS, PAGE_STATUS_QUESTION

    engine = Engine.load(str(RUN / "best.pt"))
    examples = {
        "login wall": ("https://forum.example/login", "<html><head><title>Sign in</title></head><body>"
                       "<form><input type='email' name='email'><input type='password' name='pw'>"
                       "<button>Sign in</button></form><p>" + "Members only. Please sign in to continue. " * 6
                       + "</p></body></html>"),
        "news story": ("https://city-news.example/2026/09/bridge-reopens", "<html><head><title>Bridge reopens"
                       "</title></head><body><nav><a href='/'>Home</a><a href='/city'>City</a></nav><main>"
                       "<h1>Old bridge reopens after repairs</h1><time>12 September 2026</time><p>"
                       + "Traffic returned to the river crossing on Friday after eight months of work. " * 4
                       + "</p></main><footer><a href='/privacy'>Privacy</a></footer></body></html>"),
    }
    questions = {"page_type": {"type": "choice", "instructions": PAGE_TYPE_QUESTION, "criteria": PAGE_TYPES},
                 "page_status": {"type": "choice", "instructions": PAGE_STATUS_QUESTION, "criteria": PAGE_STATUS}}
    for name, (url, html) in examples.items():
        answers = engine.predict(page_state(html, url), questions)["answers"]
        print(name, {q: (a["choice"], a["confidence"]) for q, a in answers.items()})
except Exception as exc:   # optional step: never block saving
    print('demo prediction skipped:', repr(exc))

## 8. Export to ONNX (CPU inference, no torch)

In [ ]:
try:
    temps = json.dumps(report["temperature_by_type"])
    sh(f"python scripts/export_onnx.py --checkpoint {RUN}/best.pt --output {RELEASE} --data {DATA} "
       f"--name 'Anarkali Web 150M' --temperature-by-type '{temps}' --no-int8")
    shutil.copy(RUN / "test-report.json", RELEASE / "test-report.json")
    print(sorted(p.name for p in RELEASE.iterdir()))
except Exception as exc:   # optional step: never block saving
    print('ONNX export skipped:', repr(exc))


## 9. Save everything (best.pt included)

In [ ]:
OUT = WORK / "anarkali-web-output"
OUT.mkdir(exist_ok=True)
shutil.copy(RUN / "best.pt", OUT / "best.pt")
for name in ("training.json", "test-report.json", "anarkali-test.jsonl"):
    shutil.copy(RUN / name, OUT / name)
shutil.copy(DATA / "manifest.json", OUT / "dataset-manifest.json")
shutil.copy(LOG, OUT / "notebook-log.txt")
if julia_result is not None:
    (OUT / "julia-baseline.json").write_text(json.dumps(julia_result, indent=1))
hf.upload_folder(repo_id=HF_REPO, folder_path=str(OUT), path_in_repo=f"{RUN_NAME}/web-run",
                 allow_patterns=["*.json", "*.jsonl", "*.txt"])
print("reports uploaded to", f"{HF_REPO}/{RUN_NAME}/web-run")
if RELEASE.exists():
    shutil.make_archive(str(WORK / "release-web"), "zip", RELEASE)
print("saved:", OUT, "and", WORK / "release-web.zip")

if not ON_KAGGLE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        target = Path("/content/drive/MyDrive/anarkali-web")
        shutil.copytree(OUT, target / "run", dirs_exist_ok=True)
        shutil.copy(WORK / "release-web.zip", target / "release-web.zip")
        print("copied to Google Drive:", target)
    except Exception as exc:
        print("Drive copy skipped:", exc)

In [ ]:
try:
    # Optional: the ONNX release too (CPU inference without torch).
    hf.upload_folder(repo_id=HF_REPO, folder_path=str(RELEASE), path_in_repo=f"{RUN_NAME}/release-web")
    print("ONNX release uploaded to", f"{HF_REPO}/{RUN_NAME}/release-web")
except Exception as exc:   # optional step: never block saving
    print('ONNX upload skipped:', repr(exc))
